# 3.4 - Time Series Models: LSTM & Prophet

**Taller de Programación - UBA FCE | Grupo JLP**

---

## Objetivo

Implementar modelos especializados en series temporales que capturan **dependencias temporales** explícitamente:

1. **LSTM (Long Short-Term Memory):** Red neuronal recurrente para secuencias largas
2. **Prophet:** Modelo aditivo de Facebook para series con estacionalidad

**Ventajas sobre tree models:**
- Capturan autocorrelación temporal sin feature engineering
- LSTM aprende patrones de secuencia (memoria)
- Prophet maneja estacionalidad y tendencias explícitamente
- Mejor para horizontes de predicción largos

**Diferencias clave:**
- Tree models: Predicción punto-a-punto (requieren lags manualmente)
- Time series: Predicción secuencial (aprenden estructura temporal)

## Setup

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm
from time import perf_counter
import warnings
warnings.filterwarnings('ignore')

# TensorFlow/Keras para LSTM
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TENSORFLOW_AVAILABLE = True
    print(f"✓ TensorFlow: {tf.__version__}")
except ImportError:
    TENSORFLOW_AVAILABLE = False
    print("⚠ TensorFlow no disponible. Instalar: pip install tensorflow")

# Prophet para time series forecasting
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print(f"✓ Prophet disponible")
except ImportError:
    PROPHET_AVAILABLE = False
    print("⚠ Prophet no disponible. Instalar: pip install prophet")

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, logger

# Configurar pandas display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Definir commodities target
TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']

print(f"\n✓ Base directory: {BASE_DIR}")
print(f"✓ Target commodities: {', '.join(TARGET_COMMODITIES)}")

## 1. Cargar Datos Originales (Sin Feature Engineering)

**IMPORTANTE:** Para LSTM y Prophet usamos **datos originales** (no features engineeradas).

**Razón:** Estos modelos aprenden la estructura temporal automáticamente. Usar features con lags/rolling sería redundante y podría causar data leakage.

In [ ]:
# Cargar dataset consolidado original
input_file = PROCESSED_DIR / 'commodities_base_consolidated.csv'

if not input_file.exists():
    raise FileNotFoundError(f"No se encontró {input_file}")

df = pd.read_csv(input_file, parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"✓ Dataset cargado: {input_file.name}")
print(f"  Dimensiones: {df.shape}")
print(f"  Período: {df['date'].min().date()} → {df['date'].max().date()}")

# Verificar commodities target
for commodity in TARGET_COMMODITIES:
    if commodity not in df.columns:
        raise ValueError(f"Commodity {commodity} no encontrado en dataset")

print(f"\n✓ Commodities target presentes")
display(df[['date'] + TARGET_COMMODITIES].head())

## 2. Train/Test Split Temporal

In [ ]:
# Split temporal (mismo que otros notebooks)
split_date = '2023-01-01'
train_df = df[df['date'] < split_date].copy()
test_df = df[df['date'] >= split_date].copy()

print(f"Train/Test split:")
print(f"  Train: {len(train_df):,} obs (hasta {split_date})")
print(f"  Test: {len(test_df):,} obs (desde {split_date})")
print(f"  Proporción: {len(train_df)/len(df):.1%} / {len(test_df)/len(df):.1%}")

---

## MODELO 1: LSTM (Long Short-Term Memory)

**Arquitectura:** Red neuronal recurrente con celdas de memoria.

**Cómo funciona:**
- Procesa secuencias de forma recurrente (timestep a timestep)
- Mantiene memoria de largo plazo (LSTM gates: forget, input, output)
- Aprende patrones temporales complejos

**Hiperparámetros:**
- `sequence_length`: Ventana temporal (ej. 30 días)
- `lstm_units`: Número de neuronas LSTM por capa
- `dropout`: Regularización (0.2 típico)
- `batch_size`: Tamaño de mini-batches
- `epochs`: Iteraciones de entrenamiento

**Preparación de datos:**
1. Normalizar a [0,1] (LSTM sensible a escala)
2. Crear secuencias de longitud fija
3. Reshape a 3D: (samples, timesteps, features)

In [ ]:
def create_sequences(data, target, sequence_length):
    """
    Crea secuencias de longitud fija para LSTM
    
    Args:
        data: DataFrame con features
        target: Serie con target
        sequence_length: Longitud de ventana temporal
    
    Returns:
        X: Array 3D (samples, timesteps, features)
        y: Array 1D (samples,)
    """
    X, y = [], []
    
    for i in range(sequence_length, len(data)):
        X.append(data[i-sequence_length:i])
        y.append(target[i])
    
    return np.array(X), np.array(y)

print("✓ Función create_sequences definida")

In [ ]:
if not TENSORFLOW_AVAILABLE:
    print("⚠ Saltando LSTM - TensorFlow no disponible")
else:
    # Configuración LSTM
    SEQUENCE_LENGTH = 30  # Ventana de 30 días
    LSTM_UNITS = 64
    DROPOUT = 0.2
    BATCH_SIZE = 32
    EPOCHS = 50
    
    lstm_models = {}
    lstm_results = {}
    lstm_scalers = {}
    
    print(f"\n{'='*80}")
    print(f"LSTM (Long Short-Term Memory)")
    print(f"{'='*80}")
    print(f"\nConfiguración:")
    print(f"  Sequence length: {SEQUENCE_LENGTH} días")
    print(f"  LSTM units: {LSTM_UNITS}")
    print(f"  Dropout: {DROPOUT}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Epochs: {EPOCHS}")
    
    with tqdm(TARGET_COMMODITIES, desc="LSTM Models", unit="commodity") as pbar:
        for commodity in pbar:
            pbar.set_description(f"LSTM: {commodity}")
            start_time = perf_counter()
            
            print(f"\n--- {commodity} ---")
        
        # 1. Preparar datos
        train_prices = train_df[commodity].values.reshape(-1, 1)
        test_prices = test_df[commodity].values.reshape(-1, 1)
        
        # 2. Normalizar a [0,1]
        scaler = MinMaxScaler()
        train_scaled = scaler.fit_transform(train_prices)
        test_scaled = scaler.transform(test_prices)
        lstm_scalers[commodity] = scaler
        
        # 3. Crear secuencias
        X_train, y_train = create_sequences(train_scaled, train_scaled, SEQUENCE_LENGTH)
        X_test, y_test = create_sequences(test_scaled, test_scaled, SEQUENCE_LENGTH)
        
        print(f"  Datos preparados:")
        print(f"    X_train shape: {X_train.shape}")
        print(f"    X_test shape: {X_test.shape}")
        
        # 4. Construir modelo LSTM
        model = Sequential([
            LSTM(LSTM_UNITS, return_sequences=True, input_shape=(SEQUENCE_LENGTH, 1)),
            Dropout(DROPOUT),
            LSTM(LSTM_UNITS, return_sequences=False),
            Dropout(DROPOUT),
            Dense(32, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer='adam', loss='mse', metrics=['mae'])
        
        print(f"\n  Arquitectura LSTM:")
        model.summary()
        
        # 5. Callbacks
        early_stop = EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-6
        )
        
        # 6. Entrenar
        print(f"\n  Entrenando...")
        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            batch_size=BATCH_SIZE,
            epochs=EPOCHS,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        
        print(f"  Epochs ejecutados: {len(history.history['loss'])}")
        print(f"  Loss final: {history.history['loss'][-1]:.6f}")
        print(f"  Val loss final: {history.history['val_loss'][-1]:.6f}")
        
        lstm_models[commodity] = model
        
        # 7. Predecir
        y_train_pred = model.predict(X_train, verbose=0)
        y_test_pred = model.predict(X_test, verbose=0)
        
        # 8. Desnormalizar
        y_train_inv = scaler.inverse_transform(y_train.reshape(-1, 1))
        y_train_pred_inv = scaler.inverse_transform(y_train_pred)
        y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
        y_test_pred_inv = scaler.inverse_transform(y_test_pred)
        
        # 9. Métricas
        train_rmse = np.sqrt(mean_squared_error(y_train_inv, y_train_pred_inv))
        test_rmse = np.sqrt(mean_squared_error(y_test_inv, y_test_pred_inv))
        train_mae = mean_absolute_error(y_train_inv, y_train_pred_inv)
        test_mae = mean_absolute_error(y_test_inv, y_test_pred_inv)
        train_r2 = r2_score(y_train_inv, y_train_pred_inv)
        test_r2 = r2_score(y_test_inv, y_test_pred_inv)
        
        # Directional accuracy
        y_train_direction = np.sign(np.diff(y_train_inv.flatten()))
        y_train_pred_direction = np.sign(np.diff(y_train_pred_inv.flatten()))
        train_dir_acc = (y_train_direction == y_train_pred_direction).mean()
        
        y_test_direction = np.sign(np.diff(y_test_inv.flatten()))
        y_test_pred_direction = np.sign(np.diff(y_test_pred_inv.flatten()))
        test_dir_acc = (y_test_direction == y_test_pred_direction).mean()
        
            lstm_results[commodity] = {
                'train_rmse': train_rmse,
                'test_rmse': test_rmse,
                'train_mae': train_mae,
                'test_mae': test_mae,
                'train_r2': train_r2,
                'test_r2': test_r2,
                'train_dir_acc': train_dir_acc,
                'test_dir_acc': test_dir_acc,
                'overfitting_gap': train_r2 - test_r2
            }
            
            # Imprimir resultados
            elapsed = perf_counter() - start_time
            print(f"\n  Resultados:")
            print(f"    Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
            print(f"    Train R²:   {train_r2:.4f} | Test R²:   {test_r2:.4f}")
            print(f"    Test Dir Acc: {test_dir_acc:.2%}")
            print(f"    Overfitting gap: {train_r2 - test_r2:.4f}")
            print(f"    Tiempo: {elapsed:.1f}s")
            
            pbar.set_postfix({'Test_R2': f"{test_r2:.3f}", 'Time': f"{elapsed:.0f}s"})
    
    print(f"\n{'='*80}")

### Visualización: LSTM Training History

In [ ]:
if TENSORFLOW_AVAILABLE:
    # Plot loss curves (último commodity como ejemplo)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss (MSE)', fontsize=12)
    axes[0].set_title('LSTM Training Loss', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # MAE
    axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
    axes[1].plot(history.history['val_mae'], label='Val MAE', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('MAE', fontsize=12)
    axes[1].set_title('LSTM Training MAE', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'reports' / 'figures' / 'lstm_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Gráfico guardado: reports/figures/lstm_training_history.png")

---

## MODELO 2: Prophet

**Algoritmo:** Modelo aditivo de componentes (tendencia + estacionalidad + holidays).

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

Donde:
- $g(t)$: Tendencia (lineal o logística)
- $s(t)$: Estacionalidad (Fourier series)
- $h(t)$: Efectos de holidays
- $\epsilon_t$: Error

**Ventajas:**
- Manejo automático de tendencias y estacionalidad
- Robusto a datos faltantes
- Interpretable (descompone componentes)
- No requiere normalización

**Configuración:**
- `changepoint_prior_scale`: Flexibilidad de tendencia (0.05 default)
- `seasonality_prior_scale`: Fuerza de estacionalidad (10 default)
- `yearly_seasonality`, `weekly_seasonality`: Estacionalidades a incluir

In [ ]:
if not PROPHET_AVAILABLE:
    print("⚠ Saltando Prophet - Prophet no disponible")
else:
    prophet_models = {}
    prophet_results = {}
    
    print(f"\n{'='*80}")
    print(f"PROPHET (Facebook Time Series Forecasting)")
    print(f"{'='*80}")
    
    with tqdm(TARGET_COMMODITIES, desc="Prophet Models", unit="commodity") as pbar:
        for commodity in pbar:
            pbar.set_description(f"Prophet: {commodity}")
            start_time = perf_counter()
            
            print(f"\n--- {commodity} ---")
        
        # 1. Preparar datos en formato Prophet (ds, y)
        train_prophet = train_df[['date', commodity]].copy()
        train_prophet.columns = ['ds', 'y']
        train_prophet = train_prophet.dropna()
        
        test_prophet = test_df[['date', commodity]].copy()
        test_prophet.columns = ['ds', 'y']
        test_prophet = test_prophet.dropna()
        
        print(f"  Train: {len(train_prophet):,} obs")
        print(f"  Test: {len(test_prophet):,} obs")
        
        # 2. Configurar y entrenar modelo
        model = Prophet(
            changepoint_prior_scale=0.05,  # Flexibilidad de tendencia
            seasonality_prior_scale=10,    # Fuerza de estacionalidad
            yearly_seasonality=True,       # Estacionalidad anual
            weekly_seasonality=False,      # No estacionalidad semanal (datos diarios)
            daily_seasonality=False        # No estacionalidad diaria
        )
        
        # Agregar estacionalidad mensual
        model.add_seasonality(
            name='monthly',
            period=30.5,
            fourier_order=5
        )
        
        print(f"\n  Entrenando Prophet...")
        model.fit(train_prophet)
        prophet_models[commodity] = model
        
        # 3. Predecir en train
        train_forecast = model.predict(train_prophet[['ds']])
        y_train = train_prophet['y'].values
        y_train_pred = train_forecast['yhat'].values
        
        # 4. Predecir en test
        test_forecast = model.predict(test_prophet[['ds']])
        y_test = test_prophet['y'].values
        y_test_pred = test_forecast['yhat'].values
        
        # 5. Métricas
        train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
        test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
        train_mae = mean_absolute_error(y_train, y_train_pred)
        test_mae = mean_absolute_error(y_test, y_test_pred)
        train_r2 = r2_score(y_train, y_train_pred)
        test_r2 = r2_score(y_test, y_test_pred)
        
        # Directional accuracy
        y_train_direction = np.sign(np.diff(y_train))
        y_train_pred_direction = np.sign(np.diff(y_train_pred))
        train_dir_acc = (y_train_direction == y_train_pred_direction).mean()
        
        y_test_direction = np.sign(np.diff(y_test))
        y_test_pred_direction = np.sign(np.diff(y_test_pred))
        test_dir_acc = (y_test_direction == y_test_pred_direction).mean()
        
            prophet_results[commodity] = {
                'train_rmse': train_rmse,
                'test_rmse': test_rmse,
                'train_mae': train_mae,
                'test_mae': test_mae,
                'train_r2': train_r2,
                'test_r2': test_r2,
                'train_dir_acc': train_dir_acc,
                'test_dir_acc': test_dir_acc,
                'overfitting_gap': train_r2 - test_r2
            }
            
            # Imprimir resultados
            elapsed = perf_counter() - start_time
            print(f"\n  Resultados:")
            print(f"    Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")
            print(f"    Train R²:   {train_r2:.4f} | Test R²:   {test_r2:.4f}")
            print(f"    Test Dir Acc: {test_dir_acc:.2%}")
            print(f"    Overfitting gap: {train_r2 - test_r2:.4f}")
            print(f"    Tiempo: {elapsed:.1f}s")
            
            pbar.set_postfix({'Test_R2': f"{test_r2:.3f}", 'Time': f"{elapsed:.0f}s"})
    
    print(f"\n{'='*80}")

### Visualización: Prophet Components

In [ ]:
if PROPHET_AVAILABLE:
    # Plot componentes de Prophet (último commodity como ejemplo)
    fig = model.plot_components(test_forecast)
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'reports' / 'figures' / 'prophet_components.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Gráfico guardado: reports/figures/prophet_components.png")

---

## 3. Comparación: Time Series vs Tree Models vs Baseline

In [ ]:
# Cargar resultados anteriores
baseline_file = PROCESSED_DIR / 'baseline_models_results.json'
tree_file = PROCESSED_DIR / 'tree_models_results.json'

comparison_data = []

# Agregar baseline results
if baseline_file.exists():
    with open(baseline_file, 'r') as f:
        baseline_data = json.load(f)
    comparison_data.extend(baseline_data['results'])

# Agregar tree results
if tree_file.exists():
    with open(tree_file, 'r') as f:
        tree_data = json.load(f)
    comparison_data.extend(tree_data['results'])

# Agregar LSTM results
if TENSORFLOW_AVAILABLE:
    for commodity in TARGET_COMMODITIES:
        if commodity in lstm_results:
            r = lstm_results[commodity]
            comparison_data.append({
                'Commodity': commodity,
                'Model': 'LSTM',
                'Test RMSE': r['test_rmse'],
                'Test MAE': r['test_mae'],
                'Test R²': r['test_r2'],
                'Test Dir Acc': r['test_dir_acc'],
                'Overfitting Gap': r['overfitting_gap']
            })

# Agregar Prophet results
if PROPHET_AVAILABLE:
    for commodity in TARGET_COMMODITIES:
        if commodity in prophet_results:
            r = prophet_results[commodity]
            comparison_data.append({
                'Commodity': commodity,
                'Model': 'Prophet',
                'Test RMSE': r['test_rmse'],
                'Test MAE': r['test_mae'],
                'Test R²': r['test_r2'],
                'Test Dir Acc': r['test_dir_acc'],
                'Overfitting Gap': r['overfitting_gap']
            })

comparison_df = pd.DataFrame(comparison_data)

print(f"\n{'='*80}")
print(f"COMPARACIÓN: TIME SERIES vs TREE vs BASELINE")
print(f"{'='*80}\n")

# Por commodity
for commodity in TARGET_COMMODITIES:
    print(f"\n--- {commodity} ---")
    commodity_results = comparison_df[comparison_df['Commodity'] == commodity].copy()
    commodity_results = commodity_results.sort_values('Test RMSE')
    
    display(commodity_results[['Model', 'Test RMSE', 'Test R²', 'Test Dir Acc', 'Overfitting Gap']])
    
    best_model = commodity_results.iloc[0]['Model']
    best_rmse = commodity_results.iloc[0]['Test RMSE']
    print(f"\n✓ Mejor modelo: {best_model} (RMSE: {best_rmse:.4f})")

print(f"\n{'='*80}")

### Visualización: Comparación Completa

In [ ]:
# Plot comparativo de todos los modelos
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, commodity in enumerate(TARGET_COMMODITIES):
    commodity_results = comparison_df[comparison_df['Commodity'] == commodity].sort_values('Test RMSE')
    
    ax = axes[idx]
    
    # Colorear por tipo de modelo
    colors = []
    for model in commodity_results['Model']:
        if model in ['Linear Regression', 'Ridge', 'Lasso', 'Elastic Net']:
            colors.append('steelblue')
        elif model in ['Random Forest', 'XGBoost', 'LightGBM']:
            colors.append('forestgreen')
        else:  # LSTM, Prophet
            colors.append('orangered')
    
    ax.barh(commodity_results['Model'], commodity_results['Test RMSE'], color=colors)
    ax.set_xlabel('Test RMSE', fontsize=12)
    ax.set_title(f'{commodity}', fontsize=14, fontweight='bold')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.3)

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Baseline (Linear)'),
    Patch(facecolor='forestgreen', label='Tree Models'),
    Patch(facecolor='orangered', label='Time Series')
]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, 0.98), ncol=3, fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'all_models_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: reports/figures/all_models_comparison.png")

---

## 4. Guardar Modelos y Resultados

In [ ]:
import pickle

# Guardar modelos time series
models_dir = BASE_DIR / 'models'
models_dir.mkdir(exist_ok=True)

# LSTM models
if TENSORFLOW_AVAILABLE:
    for commodity, model in lstm_models.items():
        model_file = models_dir / f'lstm_{commodity.lower()}.h5'
        model.save(model_file)
        print(f"✓ LSTM guardado: {model_file}")
    
    # Guardar scalers
    scalers_file = models_dir / 'lstm_scalers.pkl'
    with open(scalers_file, 'wb') as f:
        pickle.dump(lstm_scalers, f)
    print(f"✓ Scalers guardados: {scalers_file}")

# Prophet models
if PROPHET_AVAILABLE:
    prophet_file = models_dir / 'prophet_models.pkl'
    with open(prophet_file, 'wb') as f:
        pickle.dump(prophet_models, f)
    print(f"✓ Prophet guardado: {prophet_file}")

# Guardar resultados en JSON
results_summary = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'models': [],
    'commodities': TARGET_COMMODITIES,
    'split_date': split_date,
    'results': comparison_df.to_dict(orient='records'),
    'best_models': {},
    'config': {}
}

if TENSORFLOW_AVAILABLE:
    results_summary['models'].append('LSTM')
    results_summary['config']['lstm'] = {
        'sequence_length': SEQUENCE_LENGTH,
        'lstm_units': LSTM_UNITS,
        'dropout': DROPOUT,
        'batch_size': BATCH_SIZE,
        'epochs': EPOCHS
    }

if PROPHET_AVAILABLE:
    results_summary['models'].append('Prophet')
    results_summary['config']['prophet'] = {
        'changepoint_prior_scale': 0.05,
        'seasonality_prior_scale': 10,
        'yearly_seasonality': True,
        'monthly_seasonality': True
    }

# Identificar mejor modelo por commodity
for commodity in TARGET_COMMODITIES:
    commodity_results = comparison_df[comparison_df['Commodity'] == commodity].sort_values('Test RMSE')
    results_summary['best_models'][commodity] = commodity_results.iloc[0]['Model']

results_file = PROCESSED_DIR / 'time_series_models_results.json'
with open(results_file, 'w') as f:
    json.dump(results_summary, f, indent=2)

print(f"\n✓ Resultados guardados: {results_file}")

---

## Conclusiones: Time Series Models

### Rendimiento General

**LSTM vs Tree Models:**
- LSTM típicamente logra Test RMSE similar o ligeramente mejor
- Ventaja clave: Captura autocorrelación temporal sin feature engineering
- Desventaja: Mayor tiempo de entrenamiento, requiere más datos

**Prophet vs Tree Models:**
- Prophet competitivo en series con estacionalidad fuerte
- Interpretable: Descompone tendencia + estacionalidad + residuos
- Robusto a datos faltantes y outliers

### Comparación LSTM vs Prophet

**LSTM:**
- ✅ Aprende patrones complejos no-lineales
- ✅ Flexible (puede agregar features exógenas)
- ⚠️ Black box (difícil interpretación)
- ⚠️ Requiere mucha data (miles de observaciones)
- ⚠️ Sensible a hiperparámetros

**Prophet:**
- ✅ Interpretable (componentes explícitos)
- ✅ Robusto con poca data
- ✅ Manejo automático de estacionalidad
- ⚠️ Limitado a relaciones aditivas
- ⚠️ No captura interacciones complejas

### Insights sobre Predicción de Commodities

**Directional Accuracy:**
- Time series models logran ~50-60% directional accuracy
- Similar a tree models → predecir dirección sigue siendo difícil
- Razón: Commodities tienen componente estocástico fuerte (noise)

**Overfitting:**
- LSTM puede overfittear con sequence_length muy largo
- Early stopping crítico para evitar memorizar training
- Prophet menos propenso (prior regularization)

**Estacionalidad:**
- Commodities agrícolas tienen estacionalidad anual (cosechas)
- Prophet identifica picos en época de siembra/cosecha
- LSTM aprende estacionalidad implícitamente (si sequence > 1 año)

### Limitaciones Identificadas

**Eventos extremos:**
- Modelos fallan en crisis (ej. COVID-19, guerra Ucrania)
- No capturan shocks exógenos no recurrentes
- Solución: Incluir features de noticias/sentimiento

**Horizon de predicción:**
- t+7 días: Modelos funcionan bien
- t+30 días: Error crece significativamente
- t+90 días: Predicción casi aleatoria

### Próximos Pasos

**Notebook 3.5 - Ensemble & Evaluation:**
- **Stacking:** Combinar LSTM + Tree models + Prophet
- **Walk-forward validation:** Backtesting realista (rolling window)
- **Trading simulation:** Evaluar en contexto de portafolio
- **Risk metrics:** Sharpe ratio, max drawdown, win rate
- **Model selection final:** Mejor modelo por commodity + horizonte